# Video Studio on Kaggle: any link → narrated video, as landscape and reel

Paste a link (a **GitHub repo**, an **article** or any web page, a **YouTube** video, a **PDF**) or just describe an idea, and get back a narrated explainer video with music, rendered both as **16:9 landscape** and **9:16 reel**. Everything runs on the free Kaggle GPU; no paid APIs.

| Step | Runs on |
|---|---|
| Read the source (GitHub API, web page, YouTube subtitles via yt-dlp, PDF text) | CPU |
| Write the script and storyboard (local LLM, Ollama `qwen2.5:7b`) | GPU 0 |
| Draw one image per scene (Stable Diffusion XL) and compose background music (MusicGen) | GPU 1 |
| Narration (Kokoro-82M text-to-speech) | CPU |
| Animate and render both formats in parallel (headless Chromium + ffmpeg), music ducked under the voice | CPU |

Inspired by [nexu-io/html-video](https://github.com/nexu-io/html-video) (Apache 2.0). Here a free local model only fills in a script; ready-made animated templates do the design, so results stay reliable.

| Path | What |
|---|---|
| `/` | The studio page: paste, pick options, watch progress, play and download both videos, edit the script and re-render |
| `/api/*` | JSON API |
| `/mcp` | MCP server for agents (tools: `make_video`, `get_job`, `get_storyboard`, `update_storyboard`, `render`, `list_jobs`, `delete_job`, `list_options`) |

### Before you run
1. **Accelerator:** GPU T4 x2 (one GPU works, more slowly). **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `VIDEO_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `VIDEO_UI_PASSWORD` (recommended): browser login is any username + this password; agents send `Authorization: Bearer <password>`. Without it a random password is printed below.
   - `GITHUB_TOKEN` (optional): any GitHub token, to lift the API limit of 60 repository lookups an hour.
4. **Run All.** First run: about 10 minutes (installs + ~12 GB of models). Then a 60-second video takes roughly 6–10 minutes on T4 x2 (images and rendering take most of it); 720p renders about twice as fast.

### Licences
SDXL 1.0: CreativeML OpenRAIL++ (commercial use allowed). Kokoro-82M: Apache 2.0. Qwen2.5: Apache 2.0. **MusicGen weights are CC-BY-NC 4.0 (non-commercial)**: set `MUSIC_MODEL = ""` for videos you sell or monetise, or add your own music later. Respect the licences of the pages and repos you turn into videos.

Videos are saved in `/kaggle/working/video_studio/jobs/` and disappear when the session ends: download what you want to keep. Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings
LLM_MODEL = "qwen2.5:7b"                                    # "qwen2.5:14b" writes better scripts, slower
IMAGE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"    # "" = no images (gradient backgrounds only, much faster)
IMAGE_STEPS = 25
MUSIC_MODEL = "facebook/musicgen-small"                     # "" = no music (MusicGen weights are non-commercial)
NARRATION = True                                           # Kokoro text-to-speech
RENDER_QUALITY = "1080p"                                   # or "720p" (about twice as fast); the page can override it
PORT = 7860
WORK_DIR = "/kaggle/working/video_studio"
LOG_DIR = "/kaggle/working/logs"
APP_DIR = '/kaggle/working/video_app'
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers use HTTP Basic (any username), agents send
  ``Authorization: Bearer <password>`` or ``X-Access-Token: <password>``. ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hmac
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        self._respond(h, Response(json.dumps({"error": "login required: HTTP Basic (any username) or "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8",
                                  {"WWW-Authenticate": 'Basic realm="%s", charset="UTF-8"' % self.name}))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, v) for k, v in h.headers.items() if k.lower() not in SECRET_HEADERS]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Reads any link: GitHub, web pages, YouTube, PDF
# Writes vs_source.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_source.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Turns any input into source material for the storyboard writer (standard library only).

fetch_source(text) accepts:
- a GitHub repository URL      -> description, stats, topics, languages, file tree, README
- a YouTube URL                -> title, channel, description and subtitles (via yt-dlp, no video download)
- a PDF link                   -> extracted text (pymupdf)
- any other web page / article -> title, description and readable text
- anything else                -> treated as a plain prompt
Returns {"kind", "url", "title", "text", "facts", "image_url"}.
"""
import html
import html.parser
import json
import os
import re
import shutil
import subprocess
import tempfile
import urllib.error
import urllib.parse
import urllib.request

UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36 video-studio"
MAX_TEXT = 14000
TIMEOUT = 30


def _get(url, headers=None, limit=20 << 20):
    req = urllib.request.Request(url, headers=dict({"User-Agent": UA}, **(headers or {})))
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        data = r.read(limit + 1)
        if len(data) > limit:
            raise ValueError("the page is larger than %d MB" % (limit >> 20))
        return data, r.headers.get("Content-Type", ""), r.geturl()


def _clip(text, n=MAX_TEXT):
    text = re.sub(r"\n{3,}", "\n\n", re.sub(r"[ \t]+", " ", text or "")).strip()
    return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + " ..."


def _human(n):
    n = int(n or 0)
    return "%.1fk" % (n / 1000) if n >= 1000 else str(n)


# ---------------------------------------------------------------------- GitHub
GH = re.compile(r"^https?://(?:www\.)?github\.com/([\w.-]+)/([\w.-]+?)(?:\.git)?(?:[/?#].*)?$")


def github(url):
    m = GH.match(url)
    owner, repo = m.group(1), m.group(2)
    hdr = {"Accept": "application/vnd.github+json"}
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        hdr["Authorization"] = "Bearer " + token
    api = "https://api.github.com/repos/%s/%s" % (owner, repo)
    try:
        info = json.loads(_get(api, hdr)[0])
    except urllib.error.HTTPError as e:
        if e.code == 404:
            raise ValueError("GitHub repository %s/%s not found (or private)" % (owner, repo))
        if e.code == 403:
            raise ValueError("GitHub API rate limit reached; add a GITHUB_TOKEN secret or try again later")
        raise
    langs = {}
    try:
        langs = json.loads(_get(api + "/languages", hdr)[0])
    except Exception:
        pass
    readme = ""
    try:
        readme = _get(api + "/readme", dict(hdr, Accept="application/vnd.github.raw"))[0].decode("utf-8", "replace")
    except Exception:
        pass
    tree = []
    try:
        tree = [("%s/" % x["name"]) if x["type"] == "dir" else x["name"]
                for x in json.loads(_get(api + "/contents", hdr)[0])][:40]
    except Exception:
        pass
    readme = re.sub(r"<!--.*?-->", "", readme, flags=re.S)
    readme = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", readme)                 # images
    readme = re.sub(r"<img[^>]*>", "", readme, flags=re.I)
    readme = re.sub(r"\[!\[[^\]]*\]\([^)]*\)\]\([^)]*\)", "", readme)     # badges
    total = sum(langs.values()) or 1
    top_langs = ["%s %d%%" % (k, round(100 * v / total)) for k, v in sorted(langs.items(), key=lambda kv: -kv[1])[:4]]
    facts = {"name": info.get("full_name"), "stars": _human(info.get("stargazers_count")),
             "forks": _human(info.get("forks_count")), "language": info.get("language"),
             "languages": top_langs, "license": (info.get("license") or {}).get("spdx_id"),
             "topics": info.get("topics", [])[:8], "homepage": info.get("homepage") or None,
             "created": (info.get("created_at") or "")[:10], "updated": (info.get("pushed_at") or "")[:10],
             "open_issues": info.get("open_issues_count")}
    text = "Repository: %s\nDescription: %s\nStars: %s, forks: %s\nLanguages: %s\nTopics: %s\nLicense: %s\n" \
           "Top-level files: %s\n\nREADME:\n%s" % (
               info.get("full_name"), info.get("description") or "", facts["stars"], facts["forks"],
               ", ".join(top_langs), ", ".join(facts["topics"]), facts["license"], ", ".join(tree), readme)
    return {"kind": "github", "url": info.get("html_url", url), "title": info.get("full_name") or repo,
            "description": info.get("description") or "", "text": _clip(text), "facts": facts,
            "image_url": (info.get("owner") or {}).get("avatar_url")}


# ---------------------------------------------------------------------- YouTube
YT = re.compile(r"^https?://(?:www\.|m\.)?(?:youtube\.com/(?:watch|shorts/|live/)|youtu\.be/)", re.I)


def _vtt_text(vtt):
    lines, last = [], None
    for line in vtt.splitlines():
        line = line.strip()
        if not line or "-->" in line or line.startswith(("WEBVTT", "Kind:", "Language:", "NOTE")) or line.isdigit():
            continue
        line = re.sub(r"<[^>]+>", "", html.unescape(line))
        if line != last:
            lines.append(line)
            last = line
    return " ".join(lines)


def youtube(url):
    if not shutil.which("yt-dlp"):
        raise ValueError("YouTube links need yt-dlp (installed by the notebook's install cell)")
    with tempfile.TemporaryDirectory() as d:
        p = subprocess.run(["yt-dlp", "--skip-download", "--write-info-json", "--write-auto-subs", "--write-subs",
                            "--sub-langs", "en.*,en", "--sub-format", "vtt", "--no-playlist", "-o",
                            os.path.join(d, "v.%(ext)s"), url], capture_output=True, text=True, timeout=180)
        info_path = os.path.join(d, "v.info.json")
        if not os.path.exists(info_path):
            raise ValueError("could not read that YouTube video: %s" % (p.stderr.strip().splitlines() or ["?"])[-1])
        info = json.load(open(info_path, encoding="utf-8"))
        subs = ""
        for f in sorted(os.listdir(d)):
            if f.endswith(".vtt"):
                subs = _vtt_text(open(os.path.join(d, f), encoding="utf-8", errors="replace").read())
                break
    facts = {"channel": info.get("channel") or info.get("uploader"), "views": _human(info.get("view_count")),
             "duration": "%d:%02d" % divmod(int(info.get("duration") or 0), 60), "date": info.get("upload_date")}
    text = "Video: %s\nChannel: %s\n\nDescription:\n%s\n\nTranscript:\n%s" % (
        info.get("title"), facts["channel"], info.get("description") or "", subs or "(no subtitles)")
    return {"kind": "youtube", "url": url, "title": info.get("title") or "YouTube video",
            "description": (info.get("description") or "")[:300], "text": _clip(text), "facts": facts,
            "image_url": info.get("thumbnail")}


# ---------------------------------------------------------------------- web pages
class _Text(html.parser.HTMLParser):
    SKIP = {"script", "style", "noscript", "svg", "nav", "footer", "header", "aside", "form", "button",
            "iframe", "template", "select"}
    BLOCK = {"p", "div", "section", "article", "li", "br", "h1", "h2", "h3", "h4", "h5", "h6", "pre", "blockquote",
             "tr", "td", "main", "figcaption", "dd", "dt"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.out, self.skip, self.meta, self.title, self.in_title = [], 0, {}, "", False

    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "meta":
            key = (a.get("property") or a.get("name") or "").lower()
            if key in ("og:title", "og:description", "description", "og:image", "twitter:image", "og:site_name"):
                self.meta.setdefault(key, a.get("content") or "")
        elif tag == "title":
            self.in_title = True
        elif tag in self.SKIP:
            self.skip += 1
        elif tag in self.BLOCK:
            self.out.append("\n")
            if tag in ("h1", "h2", "h3"):
                self.out.append("## ")

    def handle_endtag(self, tag):
        if tag == "title":
            self.in_title = False
        elif tag in self.SKIP:
            self.skip = max(0, self.skip - 1)
        elif tag in self.BLOCK:
            self.out.append("\n")

    def handle_data(self, data):
        if self.in_title:
            self.title += data
        elif not self.skip:
            self.out.append(data)


def page_text(raw):
    p = _Text()
    p.feed(raw)
    p.close()
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in "".join(p.out).splitlines()]
    # keep paragraphs and headings; drop short menu-like fragments
    keep = [ln for ln in lines if ln.startswith("## ") or len(ln.split()) >= 6]
    return p, "\n".join(keep)


def webpage(url):
    data, ctype, final = _get(url)
    if "pdf" in ctype.lower() or final.lower().split("?")[0].endswith(".pdf") or data[:5] == b"%PDF-":
        return pdf(final, data)
    if not any(t in ctype.lower() for t in ("html", "xml", "text")) and ctype:
        raise ValueError("unsupported content type %r at that link" % ctype)
    enc = re.search(r"charset=([\w-]+)", ctype)
    raw = data.decode(enc.group(1) if enc else "utf-8", "replace")
    p, text = page_text(raw)
    if "text/plain" in ctype.lower():
        text = raw
    title = p.meta.get("og:title") or " ".join(p.title.split()) or urllib.parse.urlparse(final).netloc
    desc = p.meta.get("og:description") or p.meta.get("description") or ""
    if len(text.split()) < 40:
        raise ValueError("that page has almost no readable text (it may need JavaScript or a login)")
    site = p.meta.get("og:site_name") or urllib.parse.urlparse(final).netloc.replace("www.", "")
    image = p.meta.get("og:image") or p.meta.get("twitter:image")
    return {"kind": "article", "url": final, "title": html.unescape(title)[:200], "description": html.unescape(desc)[:400],
            "text": _clip("Title: %s\nSite: %s\nSummary: %s\n\n%s" % (title, site, desc, text)),
            "facts": {"site": site, "words": len(text.split())},
            "image_url": urllib.parse.urljoin(final, image) if image else None}


def pdf(url, data):
    try:
        import pymupdf
    except ImportError:
        raise ValueError("PDF links need pymupdf (installed by the notebook's install cell)")
    doc = pymupdf.open(stream=data, filetype="pdf")
    text = "\n".join(page.get_text("text") for page in doc)
    title = (doc.metadata or {}).get("title") or os.path.basename(urllib.parse.urlparse(url).path) or "Document"
    pages = len(doc)
    doc.close()
    if len(text.split()) < 40:
        raise ValueError("that PDF has no text layer (scanned?)")
    return {"kind": "pdf", "url": url, "title": title[:200], "description": "", "text": _clip(text),
            "facts": {"pages": pages}, "image_url": None}


# ---------------------------------------------------------------------- entry point
def fetch_source(text):
    text = (text or "").strip()
    if not text:
        raise ValueError("give a link or describe the video")
    first = text.split()[0]
    if re.match(r"^https?://", first):
        url, extra = first, text[len(first):].strip()
        if GH.match(url):
            src = github(url)
        elif YT.match(url):
            src = youtube(url)
        else:
            src = webpage(url)
        src["instructions"] = extra             # text after the link: "focus on the install steps", ...
        return src
    return {"kind": "prompt", "url": None, "title": text[:80], "description": "", "text": _clip(text),
            "facts": {}, "image_url": None, "instructions": ""}
''')
print("wrote", os.path.join(APP_DIR, 'vs_source.py'))

In [ ]:
# Script and storyboard (local LLM), validation and fallback
# Writes vs_story.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_story.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Storyboard: the JSON the local LLM writes, its validation, and a fallback that needs no LLM.

storyboard = {
  "title": "...", "tagline": "...",
  "music_prompt": "upbeat electronic, 110 bpm ...",
  "scenes": [{"layout": "title|bullets|image|stats|code|quote|outro", "heading": "...", "bullets": [...],
              "narration": "...", "image_prompt": "...", "code": "...", "quote": "...",
              "stats": [{"value": "12.3k", "label": "stars"}]}]
}
The model never writes HTML: templates in vs_scenes.py animate whatever it puts in these fields.
"""
import json
import re
import urllib.request

LAYOUTS = ("title", "bullets", "image", "stats", "code", "quote", "outro")
LENGTHS = {30: (4, 5), 60: (6, 8), 90: (8, 11)}
WORDS_PER_SECOND = 2.4

PROMPT = """You are a video scriptwriter. Turn the SOURCE below into a {length}-second explainer video storyboard.

Return ONLY a JSON object with this shape:
{{
  "title": "short video title (max 8 words)",
  "tagline": "one-line hook (max 14 words)",
  "music_prompt": "background music description: genre, mood, tempo, instruments (no vocals)",
  "scenes": [
    {{
      "layout": one of "title", "bullets", "image", "stats", "code", "quote", "outro",
      "heading": "on-screen heading, max 7 words",
      "bullets": ["2-4 short on-screen points, max 9 words each (bullets layout only)"],
      "narration": "what the voice says during this scene: 1-3 natural sentences, max 40 words",
      "image_prompt": "a vivid illustration for this scene: subject, style, lighting; no text, no logos, no people's faces",
      "code": "a short real snippet from the source, max 8 lines (code layout only, else empty)",
      "quote": "one striking sentence from the source (quote layout only, else empty)"
    }}
  ]
}}

Rules:
- {n_min} to {n_max} scenes. The first scene uses layout "title", the last uses layout "outro".
- Use "bullets" for most middle scenes; use "image" for a visual moment, "code" only if the source has real code
  (install commands count), "quote" only for a sentence that really appears in the source.{stats_rule}
- Narration in total about {words} words, spoken, friendly and concrete. Never read URLs aloud.
- Every fact must come from the SOURCE. Do not invent numbers, names or features.
- Write in the language of the SOURCE.{extra}

SOURCE ({kind}):
{text}
"""


def build_prompt(src, length):
    n_min, n_max = LENGTHS.get(length, LENGTHS[60])
    stats_rule = ("\n- Do not add a \"stats\" scene: one with the real numbers is added automatically."
                  if src["kind"] in ("github", "youtube") else "")
    extra = ("\n- The user asked: " + src["instructions"]) if src.get("instructions") else ""
    return PROMPT.format(length=length, n_min=n_min, n_max=n_max, words=int(length * WORDS_PER_SECOND),
                         stats_rule=stats_rule, extra=extra, kind=src["kind"], text=src["text"])


def ollama_json(url, model, prompt, keep_alive="10m", timeout=600):
    body = json.dumps({"model": model, "stream": False, "format": "json", "keep_alive": keep_alive,
                       "messages": [{"role": "user", "content": prompt}],
                       "options": {"temperature": 0.6, "num_ctx": 16384}}).encode()
    req = urllib.request.Request(url.rstrip("/") + "/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        content = json.loads(r.read())["message"]["content"]
    m = re.search(r"\{.*\}", content, re.S)
    if not m:
        raise ValueError("the model did not return JSON")
    return json.loads(m.group(0))


# ---------------------------------------------------------------------- validation
def _s(v, n):
    v = re.sub(r"\s+", " ", str(v or "")).strip()
    return v if len(v) <= n else v[:n].rsplit(" ", 1)[0].rstrip(",;:") + "…"


def _words(v, n):
    w = str(v or "").split()
    return " ".join(w[:n]) + ("…" if len(w) > n else "")


def stats_for(src):
    f = src.get("facts") or {}
    if src["kind"] == "github":
        stats = [{"value": f.get("stars"), "label": "GitHub stars"}, {"value": f.get("forks"), "label": "forks"}]
        if f.get("language"):
            stats.append({"value": f["language"], "label": "main language"})
        elif f.get("license"):
            stats.append({"value": f["license"], "label": "license"})
        return [s for s in stats if s["value"]]
    if src["kind"] == "youtube":
        return [s for s in ({"value": f.get("views"), "label": "views"}, {"value": f.get("duration"), "label": "long"},
                            {"value": f.get("channel"), "label": "channel"}) if s["value"]]
    return []


def stats_narration(src):
    f = src.get("facts") or {}
    if src["kind"] == "github":
        return "%s has %s stars and %s forks on GitHub%s." % (
            (f.get("name") or "It").split("/")[-1], f.get("stars"), f.get("forks"),
            ", written mostly in %s" % f["language"] if f.get("language") else "")
    if src["kind"] == "youtube":
        return "The original video by %s has %s views." % (f.get("channel") or "its creator", f.get("views"))
    return ""


def clean(sb, src, length):
    """Validates and repairs a model storyboard; raises ValueError if nothing usable is left."""
    if not isinstance(sb, dict) or not isinstance(sb.get("scenes"), list):
        raise ValueError("storyboard must be an object with a scenes list")
    n_min, n_max = LENGTHS.get(length, LENGTHS[60])
    scenes = []
    for sc in sb["scenes"]:
        if not isinstance(sc, dict):
            continue
        layout = str(sc.get("layout") or "bullets").lower().strip()
        layout = layout if layout in LAYOUTS else "bullets"
        bullets = [_s(b, 80) for b in (sc.get("bullets") or []) if isinstance(b, (str, int, float)) and str(b).strip()][:4]
        out = {"layout": layout, "heading": _s(sc.get("heading"), 70), "bullets": bullets,
               "narration": _words(sc.get("narration"), 60), "image_prompt": _s(sc.get("image_prompt"), 300),
               "code": "\n".join(str(sc.get("code") or "").splitlines()[:10])[:600],
               "quote": _s(sc.get("quote"), 220), "stats": []}
        if layout == "bullets" and not bullets:
            out["layout"] = "image" if out["image_prompt"] else "quote"
            out["quote"] = out["quote"] or out["narration"]
        if layout == "code" and not out["code"].strip():
            out["layout"] = "bullets" if bullets else "image"
        if layout == "quote" and not out["quote"]:
            out["layout"] = "bullets" if bullets else "image"
        if layout == "stats":
            continue                                   # real numbers only: added below
        if not out["heading"] and not out["narration"]:
            continue
        scenes.append(out)
    if len(scenes) < 2:
        raise ValueError("storyboard has fewer than 2 usable scenes")
    if scenes[0]["layout"] != "title":
        scenes.insert(0, {"layout": "title", "heading": "", "bullets": [], "narration": "", "image_prompt": scenes[0]["image_prompt"],
                          "code": "", "quote": "", "stats": []})
    if scenes[-1]["layout"] != "outro":
        scenes.append({"layout": "outro", "heading": "", "bullets": [], "narration": "", "image_prompt": "",
                       "code": "", "quote": "", "stats": []})
    for sc in scenes[1:-1]:
        if sc["layout"] in ("title", "outro"):
            sc["layout"] = "bullets" if sc["bullets"] else "image"
    stats = stats_for(src)
    if stats:
        scenes.insert(min(2, len(scenes) - 1), {"layout": "stats", "heading": "By the numbers", "bullets": [],
                                                "narration": stats_narration(src), "image_prompt": "", "code": "",
                                                "quote": "", "stats": stats})
    while len(scenes) > n_max + 1:                     # keep title/outro, drop from the middle
        scenes.pop(-2)
    title = _s(sb.get("title") or src["title"], 80)
    sb2 = {"title": title, "tagline": _s(sb.get("tagline") or src.get("description"), 120),
           "music_prompt": _s(sb.get("music_prompt") or "light upbeat electronic background music, no vocals", 200),
           "scenes": scenes, "source": {k: src.get(k) for k in ("kind", "url", "title", "facts")}}
    t, o = scenes[0], scenes[-1]
    t["heading"] = t["heading"] or title
    t["narration"] = t["narration"] or sb2["tagline"] or title
    o["heading"] = o["heading"] or title
    o["narration"] = o["narration"] or "Thanks for watching."
    return sb2


def fallback(src, length):
    """A plain storyboard straight from the source, for when the model fails."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", src["text"]) if len(p.split()) >= 8]
    heads = [ln[3:].strip() for ln in src["text"].splitlines() if ln.startswith("## ")]
    n = LENGTHS.get(length, LENGTHS[60])[1] - 2
    scenes = [{"layout": "title", "heading": src["title"], "narration": src.get("description") or src["title"],
               "image_prompt": "abstract technology illustration, soft gradient light, minimal"}]
    for i, p in enumerate(paras[:n]):
        sents = re.split(r"(?<=[.!?])\s+", p)
        scenes.append({"layout": "bullets", "heading": heads[i] if i < len(heads) else "Key point %d" % (i + 1),
                       "bullets": [_s(s, 80) for s in sents[:3]], "narration": _words(" ".join(sents[:2]), 40),
                       "image_prompt": "illustration about: " + _s(sents[0], 120)})
    scenes.append({"layout": "outro", "heading": src["title"], "narration": "Thanks for watching."})
    return clean({"title": src["title"], "tagline": src.get("description"), "scenes": scenes}, src, length)


def write_storyboard(src, length, ollama_url, model, keep_alive="10m", log=None):
    prompt = build_prompt(src, length)
    last = None
    for attempt in range(2):
        try:
            return clean(ollama_json(ollama_url, model, prompt, keep_alive), src, length), None
        except Exception as e:
            last = e
            if log:
                log.warning("storyboard attempt %d failed: %s", attempt + 1, e)
    return fallback(src, length), "the model's storyboard was unusable (%s); used a plain one instead" % last


def validate_user_storyboard(sb, src, length):
    """For storyboards edited in the page or sent by an agent: same repairs, but keep their stats scenes."""
    stats = [sc for sc in sb.get("scenes", []) if isinstance(sc, dict) and sc.get("layout") == "stats"]
    out = clean(dict(sb, scenes=[sc for sc in sb.get("scenes", []) if not (isinstance(sc, dict) and sc.get("layout") == "stats")]),
                dict(src, kind="prompt"), length)
    for sc in stats:
        st = [{"value": _s(x.get("value"), 20), "label": _s(x.get("label"), 30)} for x in sc.get("stats", [])
              if isinstance(x, dict) and x.get("value")][:3]
        if st:
            out["scenes"].insert(min(2, len(out["scenes"]) - 1), {
                "layout": "stats", "heading": _s(sc.get("heading") or "By the numbers", 70), "bullets": [],
                "narration": _words(sc.get("narration"), 60), "image_prompt": "", "code": "", "quote": "", "stats": st})
    out["source"] = sb.get("source") or out["source"]
    return out
''')
print("wrote", os.path.join(APP_DIR, 'vs_story.py'))

In [ ]:
# Animated scene templates, landscape and reel, four styles
# Writes vs_scenes.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_scenes.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Animated HTML for each storyboard scene, in landscape (16:9) and reel (9:16).

Every animation is a CSS animation, so the renderer can seek it exactly: window.__seek(t) pauses all
animations at t seconds and updates captions, number count-ups and code typing. Frames are therefore
identical however slow the machine is. Sizes use vmin, so one template serves both orientations.
"""
import html
import json
import os
import re

STYLES = {
    "midnight": {"bg1": "#0b1020", "bg2": "#1b2550", "fg": "#f3f5ff", "muted": "#a9b3d6", "accent": "#7c9cff",
                 "accent2": "#ff7ac6", "card": "rgba(255,255,255,.07)", "font": "'Inter','Noto Sans',system-ui,sans-serif",
                 "head": "'Inter','Noto Sans',system-ui,sans-serif", "weight": 800},
    "paper": {"bg1": "#f6f1e7", "bg2": "#e9dfcc", "fg": "#1f1a14", "muted": "#6b5f50", "accent": "#d2552d",
              "accent2": "#2d6cd2", "card": "rgba(0,0,0,.05)", "font": "'Noto Serif','DejaVu Serif',Georgia,serif",
              "head": "'Noto Serif','DejaVu Serif',Georgia,serif", "weight": 700},
    "neon": {"bg1": "#07020f", "bg2": "#1a0638", "fg": "#f6f0ff", "muted": "#b9a6e0", "accent": "#00f0ff",
             "accent2": "#ff2bd6", "card": "rgba(0,240,255,.07)", "font": "'Inter','Noto Sans',system-ui,sans-serif",
             "head": "'Noto Sans Mono','DejaVu Sans Mono',monospace", "weight": 800},
    "swiss": {"bg1": "#ffffff", "bg2": "#f1f1f1", "fg": "#111111", "muted": "#555555", "accent": "#e3001b",
              "accent2": "#111111", "card": "#f4f4f4", "font": "'Helvetica Neue','Liberation Sans',Arial,sans-serif",
              "head": "'Helvetica Neue','Liberation Sans',Arial,sans-serif", "weight": 900},
}
SIZES = {"landscape": (1920, 1080), "reel": (1080, 1920)}

CSS = """
:root{--bg1:%(bg1)s;--bg2:%(bg2)s;--fg:%(fg)s;--muted:%(muted)s;--accent:%(accent)s;--accent2:%(accent2)s;
 --card:%(card)s;--font:%(font)s;--head:%(head)s;--dur:%(dur).3fs}
*{box-sizing:border-box}
html,body{margin:0;width:100%%;height:100%%;overflow:hidden;background:var(--bg1);color:var(--fg);font-family:var(--font)}
.stage{position:absolute;inset:0;overflow:hidden;background:radial-gradient(120%% 90%% at 20%% 10%%,var(--bg2),var(--bg1) 70%%)}
.blob{position:absolute;width:70vmin;height:70vmin;border-radius:50%%;filter:blur(9vmin);opacity:.45;animation:drift var(--dur) ease-in-out both}
.blob.a{background:var(--accent);left:-15vmin;top:-20vmin}.blob.b{background:var(--accent2);right:-20vmin;bottom:-25vmin;animation-direction:reverse}
@keyframes drift{from{transform:translate(0,0) scale(1)}to{transform:translate(12vmin,8vmin) scale(1.15)}}
.bgimg{position:absolute;inset:-2%%;background-size:cover;background-position:center;animation:kb var(--dur) linear both}
@keyframes kb{from{transform:scale(1.02)}to{transform:scale(1.13) translate(-1.5%%,-1%%)}}
.shade{position:absolute;inset:0;background:linear-gradient(180deg,rgba(0,0,0,.15),rgba(0,0,0,.72))}
.dim .shade{background:linear-gradient(180deg,rgba(0,0,0,.5),rgba(0,0,0,.72))}
.paper .shade,.swiss .shade{background:linear-gradient(180deg,rgba(255,255,255,.25),rgba(255,255,255,.88))}
.up{opacity:0;animation:up .9s cubic-bezier(.2,.75,.2,1) both}
@keyframes up{from{opacity:0;transform:translateY(5vmin)}to{opacity:1;transform:none}}
.pop{opacity:0;animation:pop .8s cubic-bezier(.2,.9,.25,1.2) both}
@keyframes pop{from{opacity:0;transform:scale(.85)}to{opacity:1;transform:none}}
.veil{position:absolute;inset:0;background:var(--bg1);pointer-events:none;z-index:50;
 animation:veilin .45s ease-out both,veilout .45s ease-in calc(var(--dur) - .45s) both}
@keyframes veilin{from{opacity:1}to{opacity:0}}@keyframes veilout{from{opacity:0}to{opacity:1}}
.first .veil{animation:veilout .45s ease-in calc(var(--dur) - .45s) both;opacity:0}
.last .veil{animation:veilin .45s ease-out both}
.progress{position:absolute;left:0;bottom:0;height:.7vmin;background:var(--accent);z-index:40;animation:prog var(--dur) linear both}
@keyframes prog{from{width:%(p0).3f%%}to{width:%(p1).3f%%}}
.content{position:absolute;inset:0;padding:8vmin 9vmin;display:flex;flex-direction:column;justify-content:center;z-index:5}
h1,h2{font-family:var(--head);font-weight:%(weight)d;letter-spacing:-.02em;margin:0;line-height:1.05}
h1{font-size:9vmin}h2{font-size:6.6vmin;margin-bottom:4vmin}
.reel h1{font-size:9.5vmin}.reel h2{font-size:7vmin}
.kicker{font:700 2.6vmin var(--font);letter-spacing:.25em;text-transform:uppercase;color:var(--accent);margin-bottom:2.4vmin}
.chip{display:inline-block;align-self:flex-start;padding:1.1vmin 2.4vmin;border-radius:99vmin;background:var(--card);
 border:.25vmin solid color-mix(in srgb,var(--accent) 60%%,transparent);color:var(--fg);font:600 3vmin var(--font);margin-bottom:3.5vmin}
.tag{font-size:3.9vmin;color:var(--muted);margin:3.5vmin 0 0;max-width:80%%;line-height:1.3}
.reel .tag{max-width:100%%}
ul{list-style:none;margin:0;padding:0}
li{font-size:4.1vmin;line-height:1.3;margin:0 0 2.6vmin;padding-left:5.5vmin;position:relative}
li::before{content:"";position:absolute;left:0;top:1.35vmin;width:2.6vmin;height:2.6vmin;border-radius:.7vmin;background:var(--accent)}
.split{flex-direction:row;align-items:center;gap:6vmin}
.split .text{flex:1.15;min-width:0}.split .pic{flex:1;height:72%%;border-radius:3vmin;overflow:hidden;position:relative;
 box-shadow:0 3vmin 7vmin rgba(0,0,0,.35)}
.reel .split{flex-direction:column-reverse;justify-content:flex-end;padding-top:10vmin}
.reel .split .pic{flex:none;width:100%%;height:46%%}.reel .split .text{flex:none;width:100%%}
.pic .bgimg{inset:0}
.lower{justify-content:flex-end;padding-bottom:16vmin}
.lower h2{font-size:7.4vmin;text-shadow:0 .5vmin 3vmin rgba(0,0,0,.45);max-width:85%%}
.reel .lower{padding-bottom:30vmin}
.paper.dim .shade,.swiss.dim .shade{background:linear-gradient(180deg,rgba(255,255,255,.7),rgba(255,255,255,.9))}
.paper .lower h2,.swiss .lower h2{text-shadow:none}
.stats{display:flex;gap:4vmin;flex-wrap:wrap}.reel .stats{flex-direction:column}
.stat{flex:1;min-width:30vmin;max-width:100%%;overflow:hidden;background:var(--card);border-radius:3vmin;padding:4.5vmin 4vmin}
.stat b{display:block;font:%(weight)d 11vmin var(--head);color:var(--accent);letter-spacing:-.03em;line-height:1}
.stat b.word{font-size:6.4vmin;line-height:1.1;overflow-wrap:anywhere}
.stat span{display:block;font-size:3.2vmin;color:var(--muted);margin-top:1.6vmin}
.win{background:#0d1117;color:#e6edf3;border-radius:2.4vmin;overflow:hidden;box-shadow:0 3vmin 7vmin rgba(0,0,0,.35)}
.win .bar{height:5vmin;background:#161b22;display:flex;align-items:center;gap:1.4vmin;padding:0 2.4vmin}
.win .bar i{width:1.8vmin;height:1.8vmin;border-radius:50%%;background:#ff5f56}.win .bar i:nth-child(2){background:#ffbd2e}.win .bar i:nth-child(3){background:#27c93f}
pre{margin:0;padding:3.6vmin 4vmin;font:3.4vmin/1.45 'Noto Sans Mono','DejaVu Sans Mono',monospace;white-space:pre-wrap;word-break:break-word;min-height:24vmin}
pre .cur{display:inline-block;width:1.4vmin;height:3.2vmin;background:var(--accent);vertical-align:-.5vmin;margin-left:.3vmin}
.qmark{font:900 22vmin/0.7 Georgia,serif;color:var(--accent);height:12vmin}
.quote{font:500 5.6vmin/1.3 var(--head);max-width:92%%}
.reel .quote{font-size:6vmin}
.src{color:var(--muted);font-size:3vmin;margin-top:4vmin}
.center{align-items:center;text-align:center}.center .chip{align-self:center}.center .tag{margin-left:auto;margin-right:auto}
.url{font:600 3.4vmin 'Noto Sans Mono','DejaVu Sans Mono',monospace;color:var(--accent);margin-top:4vmin;word-break:break-all}
.cap{position:absolute;left:50%%;bottom:6.5vmin;transform:translateX(-50%%);z-index:30;max-width:86%%;text-align:center;
 font:700 3.6vmin/1.25 var(--font);color:#fff;background:rgba(0,0,0,.62);padding:1.3vmin 2.6vmin;border-radius:1.6vmin;
 opacity:0;transition:none}
.reel .cap{bottom:15vmin;font-size:4.6vmin}
.num{position:absolute;right:5vmin;top:4.5vmin;font:700 2.4vmin var(--font);color:var(--muted);z-index:20;letter-spacing:.15em}
"""

JS = """
const CAPS = %(caps)s, COUNTS = [...document.querySelectorAll('[data-count]')], TYPE = document.querySelector('[data-type]');
const capEl = document.querySelector('.cap');
function fmtNum(v, dec){ return dec ? v.toFixed(dec) : Math.round(v).toLocaleString('en-US'); }
window.__seek = function(t){
  document.getAnimations().forEach(a => { a.pause(); a.currentTime = t * 1000; });
  if (capEl) { const c = CAPS.find(c => t >= c[0] && t < c[1]); capEl.textContent = c ? c[2] : ''; capEl.style.opacity = c ? 1 : 0; }
  COUNTS.forEach((el, i) => {
    const target = parseFloat(el.dataset.count), dec = +el.dataset.dec, f = Math.min(1, Math.max(0, (t - 0.6 - i * 0.2) / 1.4));
    el.textContent = fmtNum(target * (1 - Math.pow(1 - f, 3)), dec) + el.dataset.suffix;
  });
  if (TYPE) { const full = TYPE.dataset.type, n = Math.floor(Math.max(0, t - 0.9) * %(cps)f);
    TYPE.textContent = full.slice(0, n); const cur = document.createElement('span'); cur.className = 'cur'; TYPE.appendChild(cur); }
};
window.__ready = false;
const urls = [...document.querySelectorAll('[data-bg]')].map(e => e.dataset.bg);
Promise.all([document.fonts.ready, ...urls.map(u => new Promise(r => { const im = new Image(); im.onload = im.onerror = r; im.src = u; }))])
  .then(() => { window.__seek(0); window.__ready = true; });
"""


def esc(s):
    return html.escape(str(s or ""), quote=True)


def caption_chunks(text, start, dur, max_words):
    """[[start, end, text], ...] spread over the narration's duration, weighted by word length."""
    words = (text or "").split()
    if not words or dur <= 0:
        return []
    chunks, cur = [], []
    for w in words:
        cur.append(w)
        if len(cur) >= max_words or re.search(r"[.!?;:]$", w) and len(cur) >= 3:
            chunks.append(" ".join(cur))
            cur = []
    if cur:
        chunks.append(" ".join(cur))
    total = sum(len(c) for c in chunks)
    out, t = [], start
    for c in chunks:
        d = dur * len(c) / total
        out.append([round(t, 3), round(t + d, 3), c])
        t += d
    return out


def _num(value):
    """'12.3k' -> (12.3, 1, 'k'); 'Python' -> None."""
    m = re.fullmatch(r"\s*([\d,]+(?:\.\d+)?)\s*([kKmMbB%+]*)\s*", str(value or ""))
    if not m:
        return None
    raw = m.group(1).replace(",", "")
    return float(raw), (len(raw.split(".")[1]) if "." in raw else 0), m.group(2)


def _bg(path, cls="bgimg"):
    if not path:
        return ""
    u = "file://" + os.path.abspath(path)
    return '<div class="%s" data-bg="%s" style="background-image:url(\'%s\')"></div>' % (cls, esc(u), esc(u))


def scene_html(scene, idx, n, story, style, fmt, dur, image, captions, p0, p1):
    st = dict(STYLES.get(style, STYLES["midnight"]), dur=dur, p0=p0, p1=p1)
    lay = scene["layout"]
    src = story.get("source") or {}
    label = {"github": "GitHub · " + str(src.get("title") or ""), "youtube": "YouTube", "pdf": "Document",
             "article": str((src.get("facts") or {}).get("site") or "Article")}.get(src.get("kind"), "")
    bg = '<div class="blob a"></div><div class="blob b"></div>'
    body = ""
    h = esc(scene.get("heading"))
    if lay == "title":
        bg += _bg(image) + ('<div class="shade"></div>' if image else "")
        body = '<div class="content center">%s<h1 class="up" style="animation-delay:.3s">%s</h1>%s</div>' % (
            '<div class="chip up" style="animation-delay:.15s">%s</div>' % esc(label) if label else "", h,
            '<p class="tag up" style="animation-delay:.65s">%s</p>' % esc(story.get("tagline")) if story.get("tagline") else "")
    elif lay == "bullets":
        lis = "".join('<li class="up" style="animation-delay:%.2fs">%s</li>' % (0.7 + i * 0.45, esc(b))
                      for i, b in enumerate(scene.get("bullets") or []))
        text = '<div class="text"><div class="kicker up" style="animation-delay:.1s">%02d</div><h2 class="up" ' \
               'style="animation-delay:.25s">%s</h2><ul>%s</ul></div>' % (idx, h, lis)
        pic = '<div class="pic pop" style="animation-delay:.35s">%s</div>' % _bg(image) if image else ""
        body = '<div class="content %s">%s%s</div>' % ("split" if image else "", text, pic)
    elif lay == "image":
        bg += _bg(image) + '<div class="shade"></div>'
        sub = (scene.get("bullets") or [""])[0]
        body = '<div class="content lower"><div class="kicker up" style="animation-delay:.2s">%02d</div><h2 class="up" ' \
               'style="animation-delay:.4s">%s</h2>%s</div>' % (idx, h, '<p class="tag up" style="animation-delay:.8s">%s</p>'
                                                                % esc(sub) if sub else "")
    elif lay == "stats":
        cards = ""
        for i, s in enumerate(scene.get("stats") or []):
            num = _num(s.get("value"))
            val = ('<b data-count="%s" data-dec="%d" data-suffix="%s">0</b>' % (num[0], num[1], esc(num[2])) if num
                   else '<b class="word">%s</b>' % esc(s.get("value")))
            cards += '<div class="stat pop" style="animation-delay:%.2fs">%s<span>%s</span></div>' % (
                0.4 + i * 0.2, val, esc(s.get("label")))
        body = '<div class="content"><h2 class="up" style="animation-delay:.15s">%s</h2><div class="stats">%s</div></div>' % (h, cards)
    elif lay == "code":
        code = scene.get("code") or ""
        body = '<div class="content"><h2 class="up" style="animation-delay:.15s">%s</h2><div class="win pop" ' \
               'style="animation-delay:.4s"><div class="bar"><i></i><i></i><i></i></div><pre data-type="%s"></pre></div></div>' % (h, esc(code))
    elif lay == "quote":
        body = '<div class="content"><div class="qmark up">&ldquo;</div><div class="quote up" style="animation-delay:.3s">%s</div>' \
               '<div class="src up" style="animation-delay:.9s">%s</div></div>' % (esc(scene.get("quote")), esc(src.get("title") or h))
    else:  # outro
        bg += _bg(image) + ('<div class="shade"></div>' if image else "")
        url = re.sub(r"^https?://(www\.)?", "", str(src.get("url") or ""))
        body = '<div class="content center"><h1 class="up" style="animation-delay:.2s">%s</h1>%s%s</div>' % (
            h, '<p class="tag up" style="animation-delay:.55s">%s</p>' % esc(scene.get("bullets", [""])[0] if scene.get("bullets") else story.get("tagline")),
            '<div class="url up" style="animation-delay:.9s">%s</div>' % esc(url) if url else "")
    cls = " ".join(filter(None, [fmt, style, "dim" if lay in ("title", "outro") else "", "first" if idx == 0 else "", "last" if idx == n - 1 else ""]))
    cps = max(18.0, len(scene.get("code") or "") / max(1.0, dur - 2.2))
    return ('<!doctype html><html class="%s"><head><meta charset="utf-8"><style>%s</style></head><body>'
            '<div class="stage">%s</div>%s<div class="num">%d / %d</div>%s<div class="progress"></div><div class="veil"></div>'
            '<script>%s</script></body></html>') % (
        cls, CSS % st, bg, body, idx + 1, n, '<div class="cap"></div>' if captions else "",
        JS % {"caps": json.dumps(captions or []), "cps": cps})
''')
print("wrote", os.path.join(APP_DIR, 'vs_scenes.py'))

In [ ]:
# Images (SDXL), narration (Kokoro), music (MusicGen), audio mix
# Writes vs_media.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_media.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Images, narration and music for Video Studio, plus the audio mix. Heavy libraries load on first use.

- Images: Stable Diffusion XL (diffusers), one square image per scene, cropped by the templates for 16:9 and 9:16.
- Narration: Kokoro-82M (Apache 2.0), 24 kHz.
- Music: MusicGen (transformers), ~30 s generated once, looped under the whole video.
- mix(): narration track + music ducked under the voice -> one 44.1 kHz WAV for both formats.
"""
import os
import threading
import wave

import numpy as np

STYLE_SUFFIX = {
    "midnight": "cinematic digital illustration, deep blue and magenta lighting, soft glow, high detail",
    "paper": "warm editorial illustration, paper texture, muted earthy colors, gouache",
    "neon": "cyberpunk illustration, glowing cyan and magenta neon, dark background, high contrast",
    "swiss": "minimal flat geometric illustration, bold red black and white, clean shapes, bauhaus poster",
}
NEGATIVE = "text, letters, words, watermark, logo, signature, caption, ui, blurry, lowres, deformed, extra fingers"
VOICES = {
    "af_heart": "US English, female (warm)", "af_bella": "US English, female (bright)",
    "am_michael": "US English, male", "am_fenrir": "US English, male (deep)",
    "bf_emma": "UK English, female", "bm_george": "UK English, male",
    "ef_dora": "Spanish, female", "ff_siwis": "French, female", "hf_alpha": "Hindi, female",
    "if_sara": "Italian, female", "pf_dora": "Portuguese (BR), female",
}
OUT_RATE = 44100


def read_wav(path):
    with wave.open(path) as w:
        rate, ch, n = w.getframerate(), w.getnchannels(), w.getnframes()
        data = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float32) / 32768.0
    if ch > 1:
        data = data.reshape(-1, ch).mean(axis=1)
    return data, rate


def write_wav(path, data, rate):
    data = np.clip(np.asarray(data, dtype=np.float32), -1, 1)
    tmp = path + ".tmp.wav"
    with wave.open(tmp, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes((data * 32767).astype(np.int16).tobytes())
    os.replace(tmp, path)


def resample(x, src, dst):
    if src == dst or not len(x):
        return x
    n = int(round(len(x) * dst / src))
    return np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)


def duration(path):
    with wave.open(path) as w:
        return w.getnframes() / float(w.getframerate())


class Media:
    def __init__(self, image_model, image_steps=25, image_device="cuda:0", music_model="", music_device="cuda:0",
                 tts_device="cpu", log=None):
        self.image_model, self.image_steps, self.image_device = image_model, int(image_steps), image_device
        self.music_model, self.music_device, self.tts_device = music_model, music_device, tts_device
        self.log = log
        self.lock = threading.Lock()
        self.m = {}

    # ------------------------------------------------------------------ images
    def _pipe(self):
        if "sd" not in self.m:
            import torch
            from diffusers import AutoPipelineForText2Image
            kw = {"torch_dtype": torch.float16}
            if "xl" in self.image_model.lower() and "turbo" not in self.image_model.lower():
                from diffusers import AutoencoderKL
                kw["vae"] = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
            try:
                pipe = AutoPipelineForText2Image.from_pretrained(self.image_model, variant="fp16", **kw)
            except Exception:
                pipe = AutoPipelineForText2Image.from_pretrained(self.image_model, **kw)
            pipe.set_progress_bar_config(disable=True)
            self.m["sd"] = pipe.to(self.image_device)
            if self.log:
                self.log.info("loaded %s on %s", self.image_model, self.image_device)
        return self.m["sd"]

    def image(self, prompt, style, path, seed=0):
        import torch
        with self.lock:
            pipe = self._pipe()
            full = "%s, %s" % (prompt, STYLE_SUFFIX.get(style, ""))
            gen = torch.Generator(self.image_device).manual_seed(int(seed))
            turbo = "turbo" in self.image_model.lower() or "schnell" in self.image_model.lower()
            out = pipe(prompt=full, negative_prompt=None if turbo else NEGATIVE, num_inference_steps=self.image_steps,
                       guidance_scale=0.0 if turbo else 6.0, width=1024, height=1024, generator=gen).images[0]
        out.save(path + ".tmp.png")
        os.replace(path + ".tmp.png", path)
        return path

    # ------------------------------------------------------------------ narration
    def tts(self, text, voice, path, speed=1.0):
        """Writes a 24 kHz WAV and returns its duration in seconds."""
        lang = voice[0]
        with self.lock:
            key = "tts_" + lang
            if key not in self.m:
                from kokoro import KPipeline
                try:
                    self.m[key] = KPipeline(lang_code=lang, device=self.tts_device)
                except TypeError:
                    self.m[key] = KPipeline(lang_code=lang)
            parts = []
            for r in self.m[key](text, voice=voice, speed=speed):
                audio = r.audio if hasattr(r, "audio") else r[2]
                if audio is not None:
                    parts.append(audio.detach().cpu().numpy() if hasattr(audio, "detach") else np.asarray(audio))
        if not parts:
            raise RuntimeError("no audio produced")
        data = np.concatenate(parts).astype(np.float32)
        write_wav(path, data, 24000)
        return len(data) / 24000.0

    # ------------------------------------------------------------------ music
    def music(self, prompt, path, seconds=30):
        with self.lock:
            if "mg" not in self.m:
                import torch
                from transformers import AutoProcessor, MusicgenForConditionalGeneration
                proc = AutoProcessor.from_pretrained(self.music_model)
                model = MusicgenForConditionalGeneration.from_pretrained(self.music_model, torch_dtype=torch.float16
                                                                         if self.music_device.startswith("cuda") else torch.float32)
                self.m["mg"] = (proc, model.to(self.music_device))
            proc, model = self.m["mg"]
            inputs = proc(text=[prompt + ", instrumental, no vocals, loopable"], padding=True, return_tensors="pt").to(self.music_device)
            out = model.generate(**inputs, max_new_tokens=int(min(30, seconds) * 50), do_sample=True, guidance_scale=3.0)
            rate = model.config.audio_encoder.sampling_rate
            data = out[0, 0].float().cpu().numpy()
        write_wav(path, data / (np.abs(data).max() + 1e-6) * 0.9, rate)
        return path

    def unload(self, *names):
        for n in names or list(self.m):
            self.m.pop(n, None)
        try:
            import gc
            import torch
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass


def mix(total, narrations, music_path, out_path, music_gain=0.22, duck_gain=0.07):
    """narrations: [(start_seconds, wav_path)]. Music loops with crossfades and dips under the voice."""
    n = int(total * OUT_RATE)
    voice = np.zeros(n, np.float32)
    for start, p in narrations:
        x, r = read_wav(p)
        x = resample(x, r, OUT_RATE)
        i = int(start * OUT_RATE)
        x = x[:max(0, n - i)]
        voice[i:i + len(x)] += x
    out = voice.copy()
    if music_path and os.path.exists(music_path):
        m, r = read_wav(music_path)
        m = resample(m, r, OUT_RATE)
        xf = min(int(1.5 * OUT_RATE), len(m) // 4)
        if len(m) > 2 * xf:
            loop = m.copy()
            while len(loop) < n:                         # append with a crossfade so the loop seam is smooth
                fade = np.linspace(0, 1, xf, dtype=np.float32)
                loop[-xf:] = loop[-xf:] * (1 - fade) + m[:xf] * fade
                loop = np.concatenate([loop, m[xf:]])
            m = loop[:n]
        else:
            m = np.resize(m, n)
        env = np.abs(voice)
        win = int(0.35 * OUT_RATE)
        env = np.convolve(env, np.ones(win, np.float32) / win, mode="same")
        talking = np.clip(env / 0.02, 0, 1)
        gain = music_gain - (music_gain - duck_gain) * talking
        fade_in, fade_out = int(1.0 * OUT_RATE), int(2.5 * OUT_RATE)
        gain[:fade_in] *= np.linspace(0, 1, len(gain[:fade_in]))
        gain[-fade_out:] *= np.linspace(1, 0, len(gain[-fade_out:]))
        out = out + m * gain
    peak = np.abs(out).max()
    if peak > 0.97:
        out = out * (0.97 / peak)
    write_wav(out_path, out, OUT_RATE)
    return out_path
''')
print("wrote", os.path.join(APP_DIR, 'vs_media.py'))

In [ ]:
# Renderer: headless Chromium frames -> ffmpeg
# Writes vs_render.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_render.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Renders one format of a job to MP4. Run as its own process (the server starts one per format, in parallel):
    python vs_render.py <job_dir> <landscape|reel>

Reads <job_dir>/plan.json (scenes with durations, images, captions; written by vs_server.py), writes one
HTML file per scene, seeks every frame in headless Chromium, pipes JPEG screenshots into ffmpeg, joins the
scenes and muxes <job_dir>/audio.wav. Progress goes to <job_dir>/<fmt>/progress.json.
"""
import json
import os
import shutil
import subprocess
import sys
import time

import vs_scenes as V


def progress(out_dir, **kw):
    tmp = os.path.join(out_dir, "progress.json.tmp")
    with open(tmp, "w") as f:
        json.dump(dict(kw, updated=time.time()), f)
    os.replace(tmp, os.path.join(out_dir, "progress.json"))


def ffmpeg(*args):
    p = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", *args],
                       capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("ffmpeg failed: " + p.stderr[-1500:])


def main(job_dir, fmt):
    plan = json.load(open(os.path.join(job_dir, "plan.json"), encoding="utf-8"))
    out_dir = os.path.join(job_dir, fmt)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir)
    fps = int(plan.get("fps", 30))
    w, h = V.SIZES[fmt]
    if plan.get("quality") == "720p":
        w, h = w * 2 // 3, h * 2 // 3
    scenes = plan["scenes"]
    total_frames = sum(max(1, round(s["dur"] * fps)) for s in scenes)
    total_dur = sum(s["dur"] for s in scenes)
    done, t_acc = 0, 0.0
    progress(out_dir, state="starting", frames=0, total_frames=total_frames)
    from playwright.sync_api import sync_playwright
    t0 = time.time()
    with sync_playwright() as pw:
        browser = pw.chromium.launch(executable_path=os.environ.get("CHROMIUM_PATH") or None,
                                     args=["--no-sandbox", "--disable-dev-shm-usage", "--font-render-hinting=none"])
        page = browser.new_page(viewport={"width": w, "height": h}, device_scale_factor=1)
        parts = []
        for i, sc in enumerate(scenes):
            caps = V.caption_chunks(sc["narration"], sc["narr_start"], sc["narr_dur"],
                                    5 if fmt == "reel" else 9) if plan.get("captions") else None
            doc = V.scene_html(sc, i, len(scenes), plan["story"], plan["style"], fmt, sc["dur"], sc.get("image"),
                               caps, 100 * t_acc / total_dur, 100 * (t_acc + sc["dur"]) / total_dur)
            t_acc += sc["dur"]
            path = os.path.join(out_dir, "scene_%02d.html" % i)
            with open(path, "w", encoding="utf-8") as f:
                f.write(doc)
            page.goto("file://" + os.path.abspath(path))
            page.wait_for_function("window.__ready === true", timeout=60000)
            n = max(1, round(sc["dur"] * fps))
            mp4 = os.path.join(out_dir, "scene_%02d.mp4" % i)
            enc = subprocess.Popen(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "image2pipe",
                                    "-framerate", str(fps), "-c:v", "mjpeg", "-i", "-", "-c:v", "libx264",
                                    "-preset", "veryfast", "-crf", "19", "-pix_fmt", "yuv420p", "-r", str(fps), mp4],
                                   stdin=subprocess.PIPE, stderr=subprocess.PIPE)
            try:
                for k in range(n):
                    page.evaluate("t => window.__seek(t)", k / fps)
                    enc.stdin.write(page.screenshot(type="jpeg", quality=92))
                    done += 1
                    if done % 15 == 0:
                        el = time.time() - t0
                        progress(out_dir, state="rendering", scene=i + 1, scenes=len(scenes), frames=done,
                                 total_frames=total_frames, eta=round(el / done * (total_frames - done)))
                    if i == 0 and k == min(n - 1, int(1.8 * fps)):
                        page.screenshot(path=os.path.join(out_dir, "poster.jpg"), type="jpeg", quality=88)
            finally:
                enc.stdin.close()
                err = enc.stderr.read().decode("utf-8", "replace")
                if enc.wait() != 0:
                    raise RuntimeError("ffmpeg could not encode scene %d: %s" % (i + 1, err[-800:]))
            parts.append(mp4)
        browser.close()
    progress(out_dir, state="joining", frames=done, total_frames=total_frames)
    lst = os.path.join(out_dir, "parts.txt")
    with open(lst, "w") as f:
        f.writelines("file '%s'\n" % os.path.abspath(p) for p in parts)
    silent = os.path.join(out_dir, "silent.mp4")
    ffmpeg("-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", silent)
    final = os.path.join(job_dir, "%s.mp4" % fmt)
    audio = os.path.join(job_dir, "audio.wav")
    if os.path.exists(audio):
        ffmpeg("-i", silent, "-i", audio, "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
               "-shortest", "-movflags", "+faststart", final + ".tmp.mp4")
    else:
        ffmpeg("-i", silent, "-c", "copy", "-movflags", "+faststart", final + ".tmp.mp4")
    os.replace(final + ".tmp.mp4", final)
    for p in parts + [silent, lst]:
        os.remove(p)
    if os.path.exists(os.path.join(out_dir, "poster.jpg")):
        shutil.copy(os.path.join(out_dir, "poster.jpg"), os.path.join(job_dir, "%s.jpg" % fmt))
    progress(out_dir, state="done", frames=done, total_frames=total_frames, seconds=round(time.time() - t0),
             size=os.path.getsize(final))


if __name__ == "__main__":
    try:
        main(sys.argv[1], sys.argv[2])
    except Exception as e:
        import traceback
        traceback.print_exc()
        d = os.path.join(sys.argv[1], sys.argv[2])
        os.makedirs(d, exist_ok=True)
        progress(d, state="error", error="%s: %s" % (type(e).__name__, str(e)[:800]))
        sys.exit(1)
''')
print("wrote", os.path.join(APP_DIR, 'vs_render.py'))

In [ ]:
# Studio server on port 7860: jobs, API and MCP
# Writes vs_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Video Studio: any link or idea -> narrated video with music, as landscape (16:9) and reel (9:16) MP4s.

Runs as its own process (started by the notebook with studio_http.spawn):
    APP_PASSWORD=... python vs_server.py --port 7860

Pipeline per job (one job at a time, files in WORK_DIR/jobs/<id>/):
    source (vs_source) -> storyboard (local LLM via Ollama, vs_story) -> [optional review/edit]
    -> scene images (SDXL) -> narration (Kokoro) -> music (MusicGen) -> mix (vs_media)
    -> render landscape + reel in parallel (vs_render, headless Chromium + ffmpeg)
Images and narration are cached by content, so editing one scene and re-rendering only redoes that scene.
"""
import argparse
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import uuid

import studio_http as K
import vs_media as M
import vs_scenes as V
import vs_source as SRC
import vs_story as ST

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("WORK_DIR", "/kaggle/working/video_studio")
JOBS = os.path.join(WORK, "jobs")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
LLM_MODEL = os.environ.get("LLM_MODEL", "qwen2.5:7b")
LLM_KEEP_ALIVE = os.environ.get("LLM_KEEP_ALIVE", "10m")
IMAGE_MODEL = os.environ.get("IMAGE_MODEL", "stabilityai/stable-diffusion-xl-base-1.0")
IMAGE_STEPS = int(os.environ.get("IMAGE_STEPS", "25"))
MUSIC_MODEL = os.environ.get("MUSIC_MODEL", "facebook/musicgen-small")
TTS_ENABLED = os.environ.get("TTS", "1") != "0"
MEDIA_DEVICE = os.environ.get("MEDIA_DEVICE", "cuda:0")
TTS_DEVICE = os.environ.get("TTS_DEVICE", "cpu")
UNLOAD_AFTER = os.environ.get("UNLOAD_AFTER", "0") == "1"
DEFAULT_QUALITY = os.environ.get("RENDER_QUALITY", "1080p")
FPS = int(os.environ.get("RENDER_FPS", "30"))
FORMATS = ("landscape", "reel")
MAX_WAIT = 85
log = K.file_logger(os.environ.get("APP_LOG", "/kaggle/working/logs/video_studio.log"), "video")

LOCK = threading.RLock()
WAKE = threading.Condition(LOCK)
QUEUE = []
MEDIA = M.Media(IMAGE_MODEL, IMAGE_STEPS, MEDIA_DEVICE, MUSIC_MODEL, MEDIA_DEVICE, TTS_DEVICE, log)


# ====================================================================== jobs on disk
def jdir(jid):
    jid = K.safe_name(jid, 40)
    d = os.path.join(JOBS, jid)
    if not jid or not os.path.isfile(os.path.join(d, "job.json")):
        raise K.HTTPError(404, "unknown job %r" % jid)
    return d


def load(jid):
    return K.read_json(os.path.join(jdir(jid), "job.json"))


def save(job, **kw):
    with LOCK:
        path = os.path.join(JOBS, job["id"], "job.json")
        cur = K.read_json(path, job) or job
        cur.update(kw, updated=time.time())
        K.write_json(path, cur)
        job.clear()
        job.update(cur)
    return job


def all_jobs():
    out = []
    if os.path.isdir(JOBS):
        for name in os.listdir(JOBS):
            j = K.read_json(os.path.join(JOBS, name, "job.json"))
            if j:
                out.append(j)
    return sorted(out, key=lambda j: -j["created"])


def files_of(jid):
    d = os.path.join(JOBS, jid)
    out = {}
    for f in ("landscape.mp4", "reel.mp4", "landscape.jpg", "reel.jpg"):
        if os.path.exists(os.path.join(d, f)):
            out[f] = "/api/jobs/%s/files/%s" % (jid, f)
    return out


def public(job, full=False):
    keys = ("id", "title", "state", "step", "progress", "created", "updated", "error", "warnings", "options",
            "duration", "render", "seconds")
    out = {k: job.get(k) for k in keys if k in job}
    out["files"] = files_of(job["id"])
    if full:
        sb = K.read_json(os.path.join(JOBS, job["id"], "storyboard.json"))
        if sb:
            out["storyboard"] = sb
        out["images"] = {str(i): "/api/jobs/%s/files/images/%s" % (job["id"], os.path.basename(p))
                         for i, p in (job.get("scene_images") or {}).items() if p}
    return out


def parse_options(b, base=None):
    o = dict(base or {"style": "midnight", "length": 60, "voice": "af_heart", "music": bool(MUSIC_MODEL),
                      "captions": True, "formats": list(FORMATS), "quality": DEFAULT_QUALITY, "review": False})
    if "style" in b and b["style"] is not None:
        if b["style"] not in V.STYLES:
            raise ValueError("style must be one of " + ", ".join(V.STYLES))
        o["style"] = b["style"]
    if b.get("length") not in (None, ""):
        L = int(b["length"])
        if L not in ST.LENGTHS:
            raise ValueError("length must be 30, 60 or 90 seconds")
        o["length"] = L
    if b.get("voice"):
        if b["voice"] not in M.VOICES and b["voice"] != "none":
            raise ValueError("voice must be one of %s, or none" % ", ".join(M.VOICES))
        o["voice"] = b["voice"]
    for k in ("music", "captions", "review"):
        if b.get(k) is not None and b.get(k) != "":
            o[k] = b[k] if isinstance(b[k], bool) else str(b[k]).lower() in ("1", "true", "yes", "on")
    if b.get("formats") is not None and b.get("formats") != "":
        f = b["formats"] if isinstance(b["formats"], list) else [x.strip() for x in str(b["formats"]).split(",")]
        if not f or any(x not in FORMATS for x in f):
            raise ValueError("formats must be a list with landscape and/or reel")
        o["formats"] = [x for x in FORMATS if x in f]
    if b.get("quality"):
        if b["quality"] not in ("1080p", "720p"):
            raise ValueError("quality must be 1080p or 720p")
        o["quality"] = b["quality"]
    if o["music"] and not MUSIC_MODEL:
        o["music"] = False
    return o


def create(source, opts):
    source = str(source or "").strip()
    if not source:
        raise ValueError("give a link (GitHub, article, YouTube, PDF...) or describe the video")
    if len(source) > 4000:
        raise ValueError("the description is longer than 4000 characters")
    jid = time.strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:5]
    d = os.path.join(JOBS, jid)
    os.makedirs(os.path.join(d, "images"))
    os.makedirs(os.path.join(d, "voice"))
    job = {"id": jid, "title": source.split()[0][:100], "input": source, "options": opts, "state": "queued",
           "step": "waiting", "progress": 0.0, "created": time.time(), "warnings": [], "approved": not opts["review"]}
    K.write_json(os.path.join(d, "job.json"), job)
    return enqueue(job)


def enqueue(job, **kw):
    with WAKE:
        save(job, state="queued", step="waiting", error=None, **kw)
        if job["id"] not in QUEUE:
            QUEUE.append(job["id"])
        WAKE.notify_all()
    return job


# ====================================================================== pipeline
def h(*parts):
    return hashlib.sha1("\x1f".join(str(p) for p in parts).encode()).hexdigest()[:16]


def scene_duration(sc, narr_dur):
    base = {"title": 3.4, "stats": 4.2, "outro": 3.6}.get(sc["layout"], 3.2)
    if sc["layout"] == "code":
        base = max(base, 2.6 + len(sc.get("code") or "") / 28.0)
    if sc["layout"] == "bullets":
        base = max(base, 1.4 + 0.45 * len(sc.get("bullets") or []))
    return round(max(base, narr_dur + 1.1), 2)


def step(job, name, frac):
    save(job, state="running", step=name, progress=round(frac, 3))
    log.info("%s: %s", job["id"], name)


def run_job(job):
    d = os.path.join(JOBS, job["id"])
    o = job["options"]
    t0 = time.time()
    warnings = []
    # 1. source
    src = K.read_json(os.path.join(d, "source.json"))
    if not src:
        step(job, "reading the source", 0.02)
        src = SRC.fetch_source(job["input"])
        K.write_json(os.path.join(d, "source.json"), src)
        save(job, title=src["title"][:100])
    # 2. storyboard
    sb = K.read_json(os.path.join(d, "storyboard.json"))
    if not sb:
        step(job, "writing the script (%s)" % LLM_MODEL, 0.06)
        try:
            sb, warn = ST.write_storyboard(src, o["length"], OLLAMA_URL, LLM_MODEL, LLM_KEEP_ALIVE, log)
        except Exception as e:
            sb, warn = ST.fallback(src, o["length"]), "script model unavailable (%s); used a plain script" % e
        if warn:
            warnings.append(warn)
        K.write_json(os.path.join(d, "storyboard.json"), sb)
        save(job, title=sb["title"][:100])
    if not job.get("approved"):
        save(job, state="review", step="review the script, then render", progress=0.15, warnings=warnings)
        return
    scenes = sb["scenes"]
    n = len(scenes)
    # 3. images
    images = {}
    if IMAGE_MODEL:
        for i, sc in enumerate(scenes):
            if sc["layout"] not in ("title", "bullets", "image", "outro") or not sc.get("image_prompt"):
                continue
            if sc["layout"] == "outro" and scenes[0].get("image_prompt"):
                sc = dict(sc, image_prompt=scenes[0]["image_prompt"])
            path = os.path.join(d, "images", h(sc["image_prompt"], o["style"], IMAGE_MODEL) + ".png")
            if not os.path.exists(path):
                step(job, "drawing image %d/%d" % (i + 1, n), 0.15 + 0.35 * i / n)
                try:
                    MEDIA.image(sc["image_prompt"], o["style"], path, seed=int(h(sc["image_prompt"])[:6], 16))
                except Exception as e:
                    log.error("image failed\n%s", traceback.format_exc())
                    warnings.append("image for scene %d failed: %s" % (i + 1, str(e)[:200]))
                    continue
            images[i] = path
    # 4. narration
    narr = {}
    if TTS_ENABLED and o["voice"] != "none":
        for i, sc in enumerate(scenes):
            text = (sc.get("narration") or "").strip()
            if not text:
                continue
            path = os.path.join(d, "voice", h(text, o["voice"]) + ".wav")
            if not os.path.exists(path):
                step(job, "recording narration %d/%d" % (i + 1, n), 0.5 + 0.12 * i / n)
                try:
                    MEDIA.tts(text, o["voice"], path)
                except Exception as e:
                    log.error("tts failed\n%s", traceback.format_exc())
                    warnings.append("narration failed (%s); the video has captions only" % str(e)[:200])
                    break
            narr[i] = (path, M.duration(path))
    # 5. timing
    plan_scenes, t = [], 0.0
    for i, sc in enumerate(scenes):
        nd = narr[i][1] if i in narr else len((sc.get("narration") or "").split()) / ST.WORDS_PER_SECOND
        dur = scene_duration(sc, nd)
        plan_scenes.append(dict(sc, dur=dur, narr_start=0.5, narr_dur=nd, image=images.get(i), start=t))
        t += dur
    total = round(t, 2)
    # 6. music
    music = None
    if o["music"] and MUSIC_MODEL:
        music = os.path.join(d, "music_%s.wav" % h(sb.get("music_prompt"), MUSIC_MODEL))
        if not os.path.exists(music):
            step(job, "composing music", 0.64)
            try:
                MEDIA.music(sb.get("music_prompt") or "light upbeat electronic", music, 30)
            except Exception as e:
                log.error("music failed\n%s", traceback.format_exc())
                warnings.append("music failed: %s" % str(e)[:200])
                music = None
    if UNLOAD_AFTER:
        MEDIA.unload()
    # 7. audio mix
    step(job, "mixing audio", 0.68)
    audio = os.path.join(d, "audio.wav")
    if narr or music:
        M.mix(total, [(ps["start"] + ps["narr_start"], narr[i][0]) for i, ps in enumerate(plan_scenes) if i in narr],
              music, audio)
    elif os.path.exists(audio):
        os.remove(audio)
    # 8. render
    K.write_json(os.path.join(d, "plan.json"), {"story": sb, "style": o["style"], "captions": o["captions"], "fps": FPS,
                                                "quality": o["quality"], "scenes": plan_scenes})
    for f in FORMATS:
        for ext in (".mp4", ".jpg"):
            if os.path.exists(os.path.join(d, f + ext)):
                os.remove(os.path.join(d, f + ext))
    procs = {f: subprocess.Popen([sys.executable, os.path.join(HERE, "vs_render.py"), d, f], cwd=HERE,
                                 stdout=open(os.path.join(d, "render_%s.log" % f), "w"), stderr=subprocess.STDOUT)
             for f in o["formats"]}
    while any(p.poll() is None for p in procs.values()):
        prog = {f: K.read_json(os.path.join(d, f, "progress.json"), {}) or {} for f in procs}
        frac = sum((p.get("frames", 0) / max(1, p.get("total_frames", 1))) for p in prog.values()) / len(procs)
        eta = max([p.get("eta", 0) or 0 for p in prog.values()] + [0])
        save(job, state="running", step="rendering %s (about %s left)" % (" + ".join(procs), "%d:%02d" % divmod(eta, 60)),
             progress=round(0.7 + 0.3 * frac, 3), render=prog)
        time.sleep(2)
    errors = []
    for f, p in procs.items():
        pr = K.read_json(os.path.join(d, f, "progress.json"), {}) or {}
        if p.returncode != 0 or pr.get("state") != "done":
            errors.append("%s: %s" % (f, pr.get("error") or K.tail(os.path.join(d, "render_%s.log" % f), 5)))
    if errors:
        raise RuntimeError("rendering failed; " + " | ".join(errors))
    save(job, state="done", step="done", progress=1.0, duration=total, warnings=warnings,
         seconds=round(time.time() - t0), scene_images={str(i): p for i, p in images.items()},
         render={f: K.read_json(os.path.join(d, f, "progress.json"), {}) for f in procs})
    log.info("%s done in %ds", job["id"], time.time() - t0)


def worker():
    while True:
        with WAKE:
            while not QUEUE:
                WAKE.wait(30)
            jid = QUEUE.pop(0)
        try:
            job = load(jid)
        except K.HTTPError:
            continue
        try:
            run_job(job)
        except Exception as e:
            log.error("job %s failed\n%s", jid, traceback.format_exc())
            save(job, state="error", step="failed", error="%s: %s" % (type(e).__name__, str(e)[:800]))


def wait(jid, seconds):
    t = time.time()
    while True:
        job = load(jid)
        if job["state"] not in ("queued", "running") or time.time() - t >= max(0, min(MAX_WAIT, float(seconds or 0))):
            return job
        time.sleep(1.5)


def set_storyboard(jid, sb, render=False):
    d = jdir(jid)
    job = load(jid)
    if job["state"] in ("queued", "running"):
        raise ValueError("the job is busy (%s); wait for it to finish" % job["state"])
    src = K.read_json(os.path.join(d, "source.json")) or {"kind": "prompt", "title": job["title"], "text": "", "facts": {}}
    if isinstance(sb, str):
        sb = json.loads(sb)
    clean = ST.validate_user_storyboard(sb, src, job["options"]["length"])
    K.write_json(os.path.join(d, "storyboard.json"), clean)
    save(job, title=clean["title"][:100])
    if render:
        enqueue(job, approved=True)
    return clean


def rerender(jid, b):
    job = load(jid)
    if job["state"] in ("queued", "running"):
        raise ValueError("the job is already %s" % job["state"])
    opts = parse_options(b or {}, job["options"])
    if b and b.get("rewrite"):
        p = os.path.join(jdir(jid), "storyboard.json")
        if os.path.exists(p):
            os.remove(p)
    return enqueue(job, options=opts, approved=not (b or {}).get("rewrite") or not opts["review"])


def delete(jid):
    job = load(jid)
    if job["state"] == "running":
        raise ValueError("the job is running; wait for it to finish")
    with LOCK:
        if jid in QUEUE:
            QUEUE.remove(jid)
    shutil.rmtree(jdir(jid))
    return {"deleted": jid}


def info():
    jobs = all_jobs()
    return {"styles": list(V.STYLES), "voices": M.VOICES, "lengths": list(ST.LENGTHS), "formats": list(FORMATS),
            "qualities": ["1080p", "720p"], "llm": LLM_MODEL, "image_model": IMAGE_MODEL or None,
            "music_model": MUSIC_MODEL or None, "tts": TTS_ENABLED, "gpus": K.gpu_info(),
            "queue": {s: sum(1 for j in jobs if j["state"] == s) for s in ("queued", "running", "review", "done", "error")}}


# ====================================================================== app
def build_app():
    app = K.App("Video Studio", password=os.environ.get("APP_PASSWORD"), log=log, max_body=8 << 20, instructions=(
        "Makes narrated explainer videos with music from any link (GitHub repo, article, YouTube, PDF) or an idea, "
        "rendered as landscape (16:9) and reel (9:16) MP4s. make_video queues a job; poll get_job until state is "
        "'done' (rendering takes a few minutes), then download files.landscape.mp4 / files.reel.mp4 with the same "
        "password. With review=true the job stops at state 'review': read get_storyboard, edit it with "
        "update_storyboard(render=true) or call render to continue."))
    app.page("/", os.path.join(HERE, "vs_ui.html"))
    app.static("/static/", HERE)

    @app.route("GET", "/api/info")
    def _info(req):
        return info()

    @app.route("GET", "/api/jobs")
    def _jobs(req):
        return {"jobs": [public(j) for j in all_jobs()]}

    @app.route("POST", "/api/jobs")
    def _create(req):
        b = req.json()
        return public(create(b.get("source"), parse_options(b)))

    @app.route("GET", r"/api/jobs/(?P<j>[\w-]+)")
    def _job(req):
        return public(wait(req.params["j"], req.arg("wait", 0, float)), full=True)

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/storyboard")
    def _sb(req):
        b = req.json()
        return {"storyboard": set_storyboard(req.params["j"], b.get("storyboard"), bool(b.get("render")))}

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/render")
    def _render(req):
        return public(rerender(req.params["j"], req.json()))

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/delete")
    def _delete(req):
        return delete(req.params["j"])

    @app.route("GET", r"/api/jobs/(?P<j>[\w-]+)/files/(?P<f>(?:images/)?[\w.-]+)")
    def _file(req):
        d = jdir(req.params["j"])
        f = req.params["f"]
        job = load(req.params["j"])
        name = "%s-%s" % (K.safe_name(job.get("title"), 50) or job["id"], f) if f.endswith(".mp4") else None
        return K.FileResponse(os.path.join(d, f), name=name if req.arg("download") else None)

    # ---------------------------------------------------------------- MCP tools
    opt_schema = {
        "style": {"type": "string", "enum": list(V.STYLES), "default": "midnight"},
        "length": {"type": "integer", "enum": list(ST.LENGTHS), "default": 60, "description": "seconds"},
        "voice": {"type": "string", "enum": list(M.VOICES) + ["none"], "default": "af_heart"},
        "music": {"type": "boolean", "default": True}, "captions": {"type": "boolean", "default": True},
        "formats": {"type": "array", "items": {"type": "string", "enum": list(FORMATS)},
                    "default": list(FORMATS)},
        "quality": {"type": "string", "enum": ["1080p", "720p"], "default": DEFAULT_QUALITY}}

    @app.tool("make_video", "Make a narrated video with music from a link (GitHub repo, article, YouTube, PDF) or a "
              "description. Text after a link is passed to the scriptwriter as instructions. Returns a job; poll get_job.",
              dict({"source": {"type": "string"}, "review": {"type": "boolean", "default": False,
                                                             "description": "stop after the script so it can be edited"},
                    "wait_seconds": {"type": "number", "default": 0}}, **opt_schema), ["source"])
    def t_make(source, wait_seconds=0, **opts):
        job = create(source, parse_options(opts))
        return public(wait(job["id"], wait_seconds) if wait_seconds else job)

    @app.tool("get_job", "Job state (queued, running, review, done, error), progress and download links. "
              "wait_seconds (max 85) waits for it to finish.", {
                  "job_id": {"type": "string"}, "wait_seconds": {"type": "number", "default": 0}}, ["job_id"])
    def t_get(job_id, wait_seconds=0):
        return public(wait(job_id, wait_seconds))

    @app.tool("get_storyboard", "The job's script: title, tagline, music prompt and scenes (layout, heading, bullets, "
              "narration, image_prompt, code, quote, stats).", {"job_id": {"type": "string"}}, ["job_id"])
    def t_sb(job_id):
        sb = K.read_json(os.path.join(jdir(job_id), "storyboard.json"))
        if not sb:
            raise ValueError("the script is not written yet (state: %s)" % load(job_id)["state"])
        return sb

    @app.tool("update_storyboard", "Replace the job's script (same shape as get_storyboard) and optionally render it. "
              "Unchanged scenes reuse their images and narration.", {
                  "job_id": {"type": "string"}, "storyboard": {"type": "object"},
                  "render": {"type": "boolean", "default": True}}, ["job_id", "storyboard"])
    def t_update(job_id, storyboard, render=True):
        return {"storyboard": set_storyboard(job_id, storyboard, render), "state": load(job_id)["state"]}

    @app.tool("render", "Render (or re-render) a job, optionally with new options. rewrite=true writes a new script first.",
              dict({"job_id": {"type": "string"}, "rewrite": {"type": "boolean", "default": False}}, **opt_schema),
              ["job_id"])
    def t_render(job_id, **b):
        return public(rerender(job_id, b))

    @app.tool("list_jobs", "Recent jobs, newest first.", {"limit": {"type": "integer", "default": 20}})
    def t_list(limit=20):
        return [public(j) for j in all_jobs()[:max(1, int(limit))]]

    @app.tool("delete_job", "Delete a job and its files.", {"job_id": {"type": "string"}}, ["job_id"])
    def t_delete(job_id):
        return delete(job_id)

    @app.tool("list_options", "Styles, voices, lengths, formats and the models in use.")
    def t_options():
        return info()

    return app


def resume():
    for j in all_jobs():
        if j["state"] in ("running", "queued"):
            QUEUE.append(j["id"])
            save(j, state="queued", step="waiting (restarted)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    os.makedirs(JOBS, exist_ok=True)
    resume()
    threading.Thread(target=worker, daemon=True).start()
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'vs_server.py'))

In [ ]:
# Studio page
# Writes vs_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Video Studio</title>
<link rel="stylesheet" href="/static/kit.css">
<style>
#source{min-height:72px;font-size:15px}
.opts{display:flex;flex-wrap:wrap;gap:12px;align-items:flex-end;margin-top:10px}
.layout{display:grid;grid-template-columns:300px 1fr;gap:16px;align-items:start}
@media (max-width:900px){.layout{grid-template-columns:1fr}}
.jobs{display:flex;flex-direction:column;gap:6px;max-height:75vh;overflow:auto}
.job{display:flex;gap:10px;align-items:center;border:1px solid var(--line);border-radius:8px;padding:6px;cursor:pointer;background:var(--panel2)}
.job.on{border-color:var(--accent)}
.job img{width:64px;height:36px;object-fit:cover;border-radius:4px;background:var(--line)}
.job .t{font-weight:600;font-size:13px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.videos{display:flex;gap:16px;flex-wrap:wrap;align-items:flex-start}
.videos .v{display:flex;flex-direction:column;gap:6px}
.videos video{background:#000;border-radius:8px;display:block}
.v.landscape video{width:min(640px,100%)}.v.reel video{height:420px;max-width:100%}
.scene{border:1px solid var(--line);border-radius:8px;padding:10px;margin-bottom:10px;display:grid;grid-template-columns:110px 1fr;gap:10px}
.scene img{width:110px;height:110px;object-fit:cover;border-radius:6px;background:var(--panel2)}
.scene .noimg{width:110px;height:110px;border-radius:6px;background:var(--panel2);display:flex;align-items:center;justify-content:center;color:var(--muted);font-size:12px}
.scene textarea{min-height:52px}
.swatch{display:inline-block;width:12px;height:12px;border-radius:3px;margin-right:4px;vertical-align:-1px}
</style></head><body>
<header class="top"><h1>Video Studio</h1><span class="sub" id="models"></span><div class="right small muted" id="status"></div></header>
<main>
<div class="card">
  <textarea id="source" placeholder="Paste a link (GitHub repo, article, YouTube video, PDF) or describe the video you want. Text after a link becomes instructions, e.g. https://github.com/owner/repo focus on the install steps"></textarea>
  <div class="opts">
    <label>Style<select id="style"></select></label>
    <label>Length<select id="length"><option value="30">30 s</option><option value="60" selected>60 s</option><option value="90">90 s</option></select></label>
    <label>Voice<select id="voice"></select></label>
    <label>Quality<select id="quality"><option>1080p</option><option>720p</option></select></label>
    <label class="inline"><input type="checkbox" id="fl" checked> Landscape 16:9</label>
    <label class="inline"><input type="checkbox" id="fr" checked> Reel 9:16</label>
    <label class="inline"><input type="checkbox" id="music" checked> Music</label>
    <label class="inline"><input type="checkbox" id="captions" checked> Captions</label>
    <label class="inline"><input type="checkbox" id="review"> Let me edit the script first</label>
    <button class="primary" id="go" style="margin-left:auto">Make video</button>
  </div>
</div>
<div class="layout">
  <div class="card"><h2>Videos</h2><div class="jobs" id="jobs"><div class="empty">Nothing yet.</div></div></div>
  <div class="card" id="detail"><div class="empty">Paste a link above and press <b>Make video</b>.</div></div>
</div>
<div class="card"><h2>For agents</h2><p class="small muted">MCP tools: make_video, get_job, get_storyboard, update_storyboard, render, list_jobs, delete_job, list_options.</p><pre class="log" id="mcp"></pre></div>
</main>
<script src="/static/kit.js"></script>
<script>
let cur = null, curState = null, INFO = null, editing = false;
$("#mcp").textContent = mcpHelp("video-studio");

function opts() {
  return {style: $("#style").value, length: +$("#length").value, voice: $("#voice").value, quality: $("#quality").value,
          music: $("#music").checked, captions: $("#captions").checked, review: $("#review").checked,
          formats: [$("#fl").checked && "landscape", $("#fr").checked && "reel"].filter(Boolean)};
}
$("#go").onclick = async () => {
  const o = opts(); if (!o.formats.length) return toast("Pick landscape, reel or both", "bad");
  const j = await guard(() => api("/api/jobs", {json: {source: $("#source").value, ...o}}), "Queued");
  cur = j.id; editing = false; loadJobs(); show();
};

async function loadInfo() {
  INFO = await api("/api/info");
  $("#models").textContent = [INFO.llm, INFO.image_model && INFO.image_model.split("/").pop(), INFO.music_model && INFO.music_model.split("/").pop()].filter(Boolean).join(" · ");
  $("#status").textContent = INFO.queue.running + " rendering · " + INFO.queue.queued + " waiting";
  if (!$("#style").options.length) {
    INFO.styles.forEach(s => $("#style").append(el("option", {value: s}, s)));
    Object.entries(INFO.voices).forEach(([k, v]) => $("#voice").append(el("option", {value: k}, v)));
    $("#voice").append(el("option", {value: "none"}, "No narration"));
    if (!INFO.music_model) { $("#music").checked = false; $("#music").disabled = true; }
  }
}

async function loadJobs() {
  const {jobs} = await api("/api/jobs");
  $("#jobs").replaceChildren(...jobs.map(j => el("div", {class: "job" + (j.id === cur ? " on" : ""), onclick: () => { cur = j.id; editing = false; loadJobs(); show(); }},
    j.files["landscape.jpg"] || j.files["reel.jpg"] ? el("img", {src: j.files["landscape.jpg"] || j.files["reel.jpg"]}) : el("img", {alt: ""}),
    el("div", {style: {minWidth: 0, flex: 1}}, el("div", {class: "t"}, j.title), el("div", {class: "row small muted"}, badge(j.state), fmtAgo(j.created)),
      j.state === "running" ? progressBar(j.progress) : null))));
  if (!jobs.length) $("#jobs").append(el("div", {class: "empty"}, "Nothing yet."));
  if (!cur && jobs.length) { cur = jobs[0].id; show(); }
  const j = jobs.find(x => x.id === cur);
  if (j && !editing && (j.state !== curState || j.state === "running" || j.state === "queued")) show();
}

function sceneEditor(sb, images) {
  return sb.scenes.map((s, i) => {
    const img = images[String(i)];
    const f = (k, v, ph, big) => el(big ? "textarea" : "input", {"data-i": i, "data-k": k, value: v || "", placeholder: ph, style: {width: "100%"}}, big ? v || "" : null);
    return el("div", {class: "scene"}, img ? el("img", {src: img}) : el("div", {class: "noimg"}, s.layout),
      el("div", {class: "col"}, el("div", {class: "row small muted"}, "Scene " + (i + 1) + " · " + s.layout),
        f("heading", s.heading, "On-screen heading"),
        s.layout === "bullets" ? f("bullets", (s.bullets || []).join("\n"), "One point per line", true) : null,
        s.layout === "code" ? f("code", s.code, "Code", true) : null,
        s.layout === "quote" ? f("quote", s.quote, "Quote", true) : null,
        f("narration", s.narration, "What the voice says", true),
        ["title", "bullets", "image", "outro"].includes(s.layout) ? f("image_prompt", s.image_prompt, "Image prompt") : null));
  });
}

function collect(sb) {
  const out = JSON.parse(JSON.stringify(sb));
  $$("[data-k]", $("#detail")).forEach(e => { const s = out.scenes[+e.dataset.i], k = e.dataset.k;
    s[k] = k === "bullets" ? e.value.split("\n").map(x => x.trim()).filter(Boolean) : e.value; });
  out.title = $("#sbtitle").value; out.tagline = $("#sbtag").value;
  return out;
}

async function show() {
  if (!cur) return;
  const j = await api("/api/jobs/" + cur);
  curState = j.state;
  const d = $("#detail"), parts = [];
  parts.push(el("div", {class: "row", style: {marginBottom: "10px"}}, el("h2", {style: {margin: 0}}, j.title), badge(j.state),
    j.duration ? el("span", {class: "small muted"}, fmtDur(j.duration) + " · " + j.options.style + " · " + j.options.quality) : null,
    el("span", {style: {marginLeft: "auto"}, class: "row"},
      ["done", "error", "review"].includes(j.state) ? el("button", {onclick: async () => { await guard(() => api("/api/jobs/" + j.id + "/render", {json: {rewrite: true, review: true}}), "Writing a new script…"); show(); }}, "New script") : null,
      el("button", {class: "danger", onclick: async () => { if (!confirm("Delete this video?")) return; await guard(() => api("/api/jobs/" + j.id + "/delete", {json: {}})); cur = null; d.replaceChildren(); loadJobs(); }}, "Delete"))));
  if (j.state === "queued" || j.state === "running") parts.push(el("div", {class: "col"}, el("div", {}, j.step), progressBar(j.progress)));
  if (j.error) parts.push(el("pre", {class: "log"}, j.error));
  (j.warnings || []).forEach(w => parts.push(el("div", {class: "small muted"}, "Note: " + w)));
  if (j.files["landscape.mp4"] || j.files["reel.mp4"]) {
    parts.push(el("div", {class: "videos", style: {margin: "10px 0"}}, ...["landscape", "reel"].filter(f => j.files[f + ".mp4"]).map(f =>
      el("div", {class: "v " + f}, el("video", {src: j.files[f + ".mp4"], controls: true, preload: "metadata", poster: j.files[f + ".jpg"] || null}),
        el("a", {class: "btn", href: j.files[f + ".mp4"] + "?download=1"}, "Download " + (f === "reel" ? "reel 9:16" : "landscape 16:9"))))));
  }
  if (j.storyboard) {
    const sb = j.storyboard;
    parts.push(el("h3", {}, j.state === "review" ? "Check the script, then render" : "Script"),
      el("div", {class: "row"}, el("label", {style: {flex: 1}}, "Title", el("input", {id: "sbtitle", value: sb.title})),
        el("label", {style: {flex: 2}}, "Tagline", el("input", {id: "sbtag", value: sb.tagline || ""}))),
      el("div", {style: {marginTop: "10px"}}, ...sceneEditor(sb, j.images || {})),
      el("div", {class: "row"},
        el("button", {class: "primary", disabled: ["queued", "running"].includes(j.state), onclick: async () => {
          editing = false; await guard(() => api("/api/jobs/" + j.id + "/storyboard", {json: {storyboard: collect(sb), render: true}}), "Rendering…"); show(); }},
          j.state === "review" ? "Render video" : "Save changes & re-render"),
        el("span", {class: "small muted"}, "Unchanged scenes keep their images and narration.")));
  }
  d.replaceChildren(...parts);
  $$("input,textarea", d).forEach(e => e.addEventListener("input", () => { editing = true; }));
}

loadInfo().then(loadJobs);
every(5000, loadInfo);
every(3000, loadJobs);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'vs_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/video_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install everything (quiet; 5-8 minutes; re-running skips finished steps)
import shutil, subprocess
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-1500:], p.stderr[-2500:])
        raise RuntimeError("failed: " + cmd[:100])

os.makedirs(LOG_DIR, exist_ok=True)
print("System packages (ffmpeg, espeak-ng, fonts)...")
sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ffmpeg espeak-ng zstd "
   "fonts-noto-core fonts-noto-mono fonts-dejavu-core > /dev/null")
print("Python packages...")
pip_install("playwright", "yt-dlp", "pymupdf", "soundfile", "diffusers", "accelerate", "kokoro>=0.9")
print("Headless Chromium...")
sh(sys.executable + " -m playwright install --with-deps chromium > /dev/null")
if not shutil.which("ollama"):
    print("Ollama...")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
K.ensure_cloudflared()

print("Downloading models (first time only)...")
from huggingface_hub import snapshot_download
if IMAGE_MODEL:
    snapshot_download(IMAGE_MODEL, allow_patterns=["*.json", "*.txt", "*fp16.safetensors", "tokenizer*/*"])
    if "xl" in IMAGE_MODEL.lower():
        snapshot_download("madebyollin/sdxl-vae-fp16-fix", allow_patterns=["*.json", "*.safetensors"])
if MUSIC_MODEL:
    snapshot_download(MUSIC_MODEL, ignore_patterns=["*.bin", "*.msgpack", "*.h5"])
if NARRATION:
    snapshot_download("hexgrad/Kokoro-82M", allow_patterns=["*.json", "*.pth", "voices/af_heart.pt", "voices/am_michael.pt"])
gpus = K.gpu_info()
print("Install complete. GPUs:", ", ".join("%d: %s" % (g["index"], g["name"]) for g in gpus) or "none - pick 'GPU T4 x2'")

In [ ]:
# 5. Start Ollama (the scriptwriter) on GPU 0 and pull the model. Log: /kaggle/working/logs/ollama.log
import subprocess
ollama_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", CUDA_VISIBLE_DEVICES="0", OLLAMA_CONTEXT_LENGTH="16384")
K.spawn("ollama", ["ollama", "serve"], LOG_DIR + "/ollama.log", env=ollama_env)
if not K.wait_http("http://127.0.0.1:11434/api/tags", timeout=60, name="ollama"):
    print(K.tail(LOG_DIR + "/ollama.log"))
    raise RuntimeError("Ollama did not start; log above")
print("Pulling %s (first time ~5 GB)..." % LLM_MODEL)
p = subprocess.run(["ollama", "pull", LLM_MODEL], env=ollama_env, capture_output=True, text=True)
if p.returncode != 0:
    print(p.stderr[-1500:])
    raise RuntimeError("ollama pull failed")
print("Scriptwriter ready.")

In [ ]:
# 6. Start the studio on port 7860 (background process). Log: /kaggle/working/logs/video_studio.log
PASSWORD = K.ui_password("VIDEO_UI_PASSWORD")
n_gpu = len(K.gpu_info())
env = dict(os.environ, APP_PASSWORD=PASSWORD, WORK_DIR=WORK_DIR, APP_LOG=LOG_DIR + "/video_studio.log",
           PYTHONUNBUFFERED="1", OLLAMA_URL="http://127.0.0.1:11434", LLM_MODEL=LLM_MODEL,
           IMAGE_MODEL=IMAGE_MODEL, IMAGE_STEPS=str(IMAGE_STEPS), MUSIC_MODEL=MUSIC_MODEL,
           TTS="1" if NARRATION else "0", TTS_DEVICE="cpu", RENDER_QUALITY=RENDER_QUALITY,
           MEDIA_DEVICE="cuda:1" if n_gpu > 1 else "cuda:0",
           LLM_KEEP_ALIVE="10m" if n_gpu > 1 else "0",      # one GPU: free the LLM's memory before drawing
           UNLOAD_AFTER="0" if n_gpu > 1 else "1", TOKENIZERS_PARALLELISM="false")
env.pop("GITHUB_TOKEN", None)
gh = K.kaggle_secret("GITHUB_TOKEN")
if gh:
    env["GITHUB_TOKEN"] = gh
K.spawn("video-studio", [sys.executable, os.path.join(APP_DIR, "vs_server.py"), "--port", str(PORT)],
        LOG_DIR + "/video_studio.log", env=env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=60, name="video-studio"):
    print(K.tail(LOG_DIR + "/video_studio.log", 40))
    raise RuntimeError("The studio did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print("Models load on the first video (1-2 minutes extra).")

In [ ]:
# 7. Publish through your Cloudflare Tunnel (token from the VIDEO_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("VIDEO_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 8. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def queue_line():
    q = K.api_get(PORT, "/api/info", PASSWORD)["queue"]
    return "videos: %d rendering, %d waiting, %d done" % (q["running"], q["queued"], q["done"])

K.keep_alive(PORT, ["video-studio", "ollama", "cloudflared"], extra=queue_line)